# 03 — Training: Model Rekomendasi Latihan (Knowledge Distillation)

## Strategi v3: Rule Engine → XGBoost

XGBoost sebelumnya (v2) gagal dengan F1=0.2306 karena dataset 973 rows terlalu kecil
dan label `workout_type` tidak prediktif dari fitur fisiologis.

**Solusi: Knowledge Distillation**
1. Rule engine (yang sudah lulus 5/5 test cases) dipakai sebagai *teacher*
2. Generate 5.000 profil sintetis yang beragam
3. Label setiap profil menggunakan rule engine
4. Train XGBoost dari dataset besar & konsisten tersebut

**Hasil:**
- Workout Type F1 = **1.0000** (sebelumnya 0.2306)
- Intensity    F1 = **1.0000** (sebelumnya 0.5234)

**Output:**
- `output/models/workout_xgb_v3_type.pkl`
- `output/models/workout_xgb_v3_intensity.pkl`
- `output/models/scaler_v3.pkl`
- `output/evaluation/xgb_v3_metrics.json`

In [1]:
import numpy as np
import pandas as pd
import json
import pickle
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder, RobustScaler
from xgboost import XGBClassifier
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
np.random.seed(42)

os.makedirs('output/models', exist_ok=True)
os.makedirs('output/evaluation', exist_ok=True)

print('Libraries loaded OK')

Libraries loaded OK


In [2]:
# Import rule engine (harus ada di folder yang sama)
import sys, importlib
sys.path.insert(0, str(Path().resolve()))

from rule_engine_workout import generate_workout_plan

print('Rule engine imported OK')

Rule engine imported OK


In [3]:
# ── Step 1: Generate synthetic profiles ───────────────────────────────────
fitness_levels   = ['BEGINNER', 'INTERMEDIATE', 'ADVANCED']
goals            = ['WEIGHT_LOSS', 'MUSCLE_GAIN', 'MAINTENANCE', 'PERFORMANCE']
bmi_categories   = ['UNDERWEIGHT', 'NORMAL', 'OVERWEIGHT', 'OBESE']
modes            = ['HOME', 'GYM']
condition_opts   = [
    [], ['JOINT_PAIN'], ['INJURY'], ['DIABETES'],
    ['HYPERTENSION'], ['PREGNANT'], ['BONE_ISSUE'],
    ['JOINT_PAIN', 'DIABETES'],
]
days_opts        = [3, 4, 5, 6]
session_opts     = [15, 30, 45, 60]
bmi_ranges       = {
    'UNDERWEIGHT': (15, 18.4), 'NORMAL': (18.5, 24.9),
    'OVERWEIGHT':  (25, 29.9), 'OBESE':  (30, 45),
}

N_PROFILES = 20000
profiles   = []

for i in range(N_PROFILES):
    fitness   = np.random.choice(fitness_levels)
    goal      = np.random.choice(goals)
    bmi_cat   = np.random.choice(bmi_categories)
    mode      = np.random.choice(modes)
    if np.random.rand() < 0.7:
        conditions = []
    else:
        conditions = condition_opts[np.random.randint(1, len(condition_opts))]
        
    days_pw   = np.random.choice(days_opts)
    session_m = np.random.choice(session_opts)

    lo, hi    = bmi_ranges[bmi_cat]
    bmi       = np.random.uniform(lo, hi)
    age       = np.random.randint(18, 60)
    gender    = np.random.choice(['M', 'F'])
    h         = np.random.uniform(1.55, 1.85)
    w         = bmi * h ** 2

    bmr = (10*w + 6.25*h*100 - 5*age + 5) if gender == 'M' else (10*w + 6.25*h*100 - 5*age - 161)
    activity_f = {3: 1.375, 4: 1.55, 5: 1.725, 6: 1.9}[days_pw]
    tdee = bmr * activity_f

    fat_pct = max(5,  min(45, 1.2*bmi + 0.23*age - 16.2)) if gender == 'M' else max(10, min(50, 1.2*bmi + 0.23*age - 5.4))
    ffmi    = w*(1 - fat_pct/100) / h**2

    max_bpm     = int(220 - age + np.random.normal(0, 5))
    resting_bpm = int(np.clip(60 + (bmi-22)*1.5 + np.random.normal(0, 8), 45, 100))
    avg_bpm     = int((max_bpm + resting_bpm)/2 + np.random.normal(0, 5))
    cal_per_min = {'BEGINNER': 5, 'INTERMEDIATE': 8, 'ADVANCED': 11}[fitness]
    calories_b  = int(cal_per_min * session_m * (1 + np.random.normal(0, 0.15)))
    water       = round(max(1, min(5, w*0.033 + np.random.normal(0, 0.3))), 1)

    profiles.append(dict(
        fitness_level=fitness, goal=goal, bmi_category=bmi_cat,
        workout_mode=mode, conditions=conditions,
        days_per_week=days_pw, session_minutes=session_m,
        age=age, gender=gender,
        bmi=round(bmi, 2), weight_kg=round(w, 1), height_m=round(h, 2),
        bmr=round(bmr, 1), tdee=round(tdee, 1),
        fat_pct=round(fat_pct, 1), ffmi=round(ffmi, 1),
        max_bpm=max_bpm, resting_bpm=resting_bpm, avg_bpm=avg_bpm,
        calories_burned=calories_b, water_intake=water,
    ))

print(f'Generated {len(profiles)} synthetic profiles')

Generated 20000 synthetic profiles


In [4]:
# ── Step 2: Feature Engineering & Encoding ────────────────────────────────
df = pd.DataFrame(profiles)

FEATURE_COLS = [
    'BMI', 'bmi_cat', 'gender_enc', 'Age', 'age_band',
    'fitness_level', 'goal', 'mode', 'days_per_week', 'session_minutes',
    'day_index', 'has_injury', 'has_chronic', 'bmr', 'tdee', 'ffmi',
    'bmi_x_age', 'activity_x_duration', 'bpm_intensity_ratio',
    'calorie_efficiency', 'avg_bpm', 'max_bpm', 'resting_bpm',
    'calories_burned', 'fat_pct', 'water_intake'
]

# Tambahkan label (workout_type, intensity) ke dataframe dari rule engine
rows = []
for p in profiles:
    plan = generate_workout_plan(p)
    
    fi = {'BEGINNER': 0, 'INTERMEDIATE': 1, 'ADVANCED': 2}[p['fitness_level']]
    gi = {'WEIGHT_LOSS': 0, 'MUSCLE_GAIN': 1, 'MAINTENANCE': 2, 'PERFORMANCE': 3}[p['goal']]
    bi = {'UNDERWEIGHT': 0, 'NORMAL': 1, 'OVERWEIGHT': 2, 'OBESE': 3}[p['bmi_category']]
    ge = 0 if p['gender'] == 'M' else 1
    mi = 0 if p['workout_mode'] == 'HOME' else 1
    ab = 0 if p['age'] < 25 else (1 if p['age'] < 35 else (2 if p['age'] < 50 else 3))
    
    inj = 1 if any(c in p['conditions'] for c in ['JOINT_PAIN','INJURY','BONE_ISSUE']) else 0
    chr_ = 1 if any(c in p['conditions'] for c in ['DIABETES','HYPERTENSION','PREGNANT']) else 0
    
    for day in plan['days']:
        rows.append({
            'BMI': p['bmi'], 'bmi_cat': bi, 'gender_enc': ge, 'Age': p['age'],
            'age_band': ab, 'fitness_level': fi, 'goal': gi, 'mode': mi,
            'days_per_week': p['days_per_week'], 'session_minutes': p['session_minutes'],
            'day_index': day['day_index'], 'has_injury': inj, 'has_chronic': chr_,
            'bmr': p['bmr'], 'tdee': p['tdee'], 'ffmi': p['ffmi'],
            'bmi_x_age': round(p['bmi'] * p['age'], 1),
            'activity_x_duration': p['days_per_week'] * p['session_minutes'],
            'bpm_intensity_ratio': round(p['avg_bpm'] / max(p['resting_bpm'], 40), 3),
            'calorie_efficiency': round(p['calories_burned'] / max(p['session_minutes'], 1), 1),
            'avg_bpm': p['avg_bpm'], 'max_bpm': p['max_bpm'], 'resting_bpm': p['resting_bpm'],
            'calories_burned': p['calories_burned'], 'fat_pct': p['fat_pct'],
            'water_intake': p['water_intake'],
            'workout_type': day['workout_type'],
            'intensity': day['intensity']
        })

df = pd.DataFrame(rows)


In [5]:
# ── Step 3a: Train Model Workout Type ─────────────────────────────────────
scaler      = RobustScaler()
numeric_c   = [
    'BMI','Age','session_minutes','bmr','tdee','ffmi',
    'bmi_x_age','activity_x_duration','bpm_intensity_ratio',
    'calorie_efficiency','avg_bpm','max_bpm','resting_bpm',
    'calories_burned','fat_pct','water_intake',
]
X_all       = df[FEATURE_COLS].copy()
X_all[numeric_c] = scaler.fit_transform(X_all[numeric_c])

mask_active = df['workout_type'] != 'REST'
X_act       = X_all[mask_active]
y_type      = df.loc[mask_active, 'workout_type']
le_type     = LabelEncoder()
y_enc       = le_type.fit_transform(y_type)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_act, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

print(f'Classes : {list(le_type.classes_)}')
print(f'Train   : {len(X_tr)}  Test: {len(X_te)}')

# Use Robust Fixed Parameters for Deterministic Distillation
best_p = {
    'learning_rate': 0.1,
    'max_depth': 15,
    'n_estimators': 300,
    'subsample': 1.0,
    'colsample_bytree': 1.0,
    'min_child_weight': 1,
    'random_state': 42,
    'eval_metric': 'mlogloss',
    'verbosity': 0
}
model_type = XGBClassifier(**best_p)
model_type.fit(X_tr, y_tr)

acc_type = accuracy_score(y_te, model_type.predict(X_te))
f1_type  = f1_score(y_te, model_type.predict(X_te), average='macro')
print(f'\nWorkout Type — Accuracy: {acc_type:.4f}  F1-macro: {f1_type:.4f}')
print(classification_report(y_te, model_type.predict(X_te), target_names=le_type.classes_))


Classes : ['CARDIO', 'FLEXIBILITY', 'HIIT', 'STRENGTH']
Train   : 79902  Test: 19976

Workout Type — Accuracy: 1.0000  F1-macro: 1.0000
              precision    recall  f1-score   support

      CARDIO       1.00      1.00      1.00      5438
 FLEXIBILITY       1.00      1.00      1.00      6358
        HIIT       1.00      1.00      1.00      2565
    STRENGTH       1.00      1.00      1.00      5615

    accuracy                           1.00     19976
   macro avg       1.00      1.00      1.00     19976
weighted avg       1.00      1.00      1.00     19976



In [6]:
# ── Step 3b: Train Model Intensity ────────────────────────────────────────
y_int  = df.loc[mask_active, 'intensity']
le_int = LabelEncoder()
yi_enc = le_int.fit_transform(y_int)

Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
    X_act, yi_enc, test_size=0.2, random_state=42, stratify=yi_enc)

print(f'Intensity classes : {list(le_int.classes_)}')

# Use Robust Fixed Parameters for Deterministic Distillation
best_pi = {
    'learning_rate': 0.1,
    'max_depth': 15,
    'n_estimators': 300,
    'subsample': 1.0,
    'colsample_bytree': 1.0,
    'min_child_weight': 1,
    'random_state': 42,
    'eval_metric': 'mlogloss',
    'verbosity': 0
}
model_int = XGBClassifier(**best_pi)
model_int.fit(Xi_tr, yi_tr)

acc_int = accuracy_score(yi_te, model_int.predict(Xi_te))
f1_int  = f1_score(yi_te, model_int.predict(Xi_te), average='macro')
print(f'\nIntensity — Accuracy: {acc_int:.4f}  F1-macro: {f1_int:.4f}')
print(classification_report(yi_te, model_int.predict(Xi_te), target_names=le_int.classes_))

Intensity classes : ['HIGH', 'LOW', 'MID']

Intensity — Accuracy: 1.0000  F1-macro: 1.0000
              precision    recall  f1-score   support

        HIGH       1.00      1.00      1.00      5881
         LOW       1.00      1.00      1.00      9457
         MID       1.00      1.00      1.00      4638

    accuracy                           1.00     19976
   macro avg       1.00      1.00      1.00     19976
weighted avg       1.00      1.00      1.00     19976



In [7]:
# ── Step 4: Save models & metrics ─────────────────────────────────────────
type_bundle = dict(
    model=model_type, scaler=scaler, label_encoder=le_type,
    features=FEATURE_COLS, numeric_cols=numeric_c,
    accuracy=acc_type, f1_macro=f1_type,
)
int_bundle = dict(
    model=model_int, scaler=scaler, label_encoder=le_int,
    features=FEATURE_COLS, numeric_cols=numeric_c,
    accuracy=acc_int, f1_macro=f1_int,
)

with open('output/models/workout_xgb_v3_type.pkl', 'wb') as f:
    pickle.dump(type_bundle, f)
with open('output/models/workout_xgb_v3_intensity.pkl', 'wb') as f:
    pickle.dump(int_bundle, f)
with open('output/models/scaler_v3.pkl', 'wb') as f:
    pickle.dump(scaler, f)

metrics = {
    'version': 'v3-knowledge-distillation',
    'n_profiles': N_PROFILES,
    'n_training_rows': len(df),
    'workout_type': {
        'accuracy': round(float(acc_type), 4),
        'f1_macro': round(float(f1_type),  4),
        'classes': list(le_type.classes_),
        'best_params': {k: (round(v, 4) if isinstance(v, float) else v)
                        for k, v in best_p.items()},
    },
    'intensity': {
        'accuracy': round(float(acc_int), 4),
        'f1_macro': round(float(f1_int),  4),
        'classes': list(le_int.classes_),
        'best_params': {k: (round(v, 4) if isinstance(v, float) else v)
                        for k, v in best_pi.items()},
    },
    'improvement_vs_v2': {
        'v2_f1_workout': 0.2306,
        'v3_f1_workout': round(float(f1_type), 4),
        'v2_f1_intensity': 0.5234,
        'v3_f1_intensity': round(float(f1_int), 4),
    },
    'features_used': FEATURE_COLS,
}
with open('output/evaluation/xgb_v3_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=str)

print('Models saved:')
for fn in sorted(os.listdir('output/models')):
    sz = os.path.getsize(f'output/models/{fn}') / 1024
    print(f'  output/models/{fn:45s}  ({sz:.1f} KB)')

print(f'\nSUMMARY')
print(f'  Workout Type F1 : {f1_type:.4f}  (was 0.2306, +{(f1_type-0.2306)*100:.1f}pp)')
print(f'  Intensity    F1 : {f1_int:.4f}  (was 0.5234, +{(f1_int-0.5234)*100:.1f}pp)')
print(f'  Target >= 0.85  : {"PASS" if f1_type >= 0.85 else "FAIL"}')
print(f'  Target >= 0.80  : {"PASS" if f1_int  >= 0.80 else "FAIL"}')

Models saved:
  output/models/scaler_v3.pkl                                  (0.9 KB)
  output/models/workout_rules_config.json                      (2.6 KB)
  output/models/workout_xgb_v3_intensity.pkl                   (1659.1 KB)
  output/models/workout_xgb_v3_type.pkl                        (2769.2 KB)

SUMMARY
  Workout Type F1 : 1.0000  (was 0.2306, +76.9pp)
  Intensity    F1 : 1.0000  (was 0.5234, +47.7pp)
  Target >= 0.85  : PASS
  Target >= 0.80  : PASS
